# 나의 감정 주치의 (Proactive AI Doctor)

> 사용자의 최근 7일 일기를 분석해 심리적 위기를 감지하고, 공감과 처방을 담은 메시지를 생성하는 파이프라인

In [1]:
## Setup

import os
import json
import numpy as np
from collections import Counter
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_teddynote import logging

load_dotenv()
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

logging.langsmith("Archive_of_feelings")
print("Setup 완료")

LangSmith 추적을 시작합니다.
[프로젝트명]
Archive_of_feelings
Setup 완료


## Stage 1 — Trigger



```
[Stage 1 - Trigger] - 로직에 대해서 추가적인 수정이 필요.

EmotionalTriggerAnalyzer 클래스 생성
 - necessity_score > 0.6 → 트리거 발동
```
`EmotionalTriggerAnalyzer`가 7일 일기를 벡터 공간에서 분석해 necessity_score를 산출합니다.
| 구성 요소 | 설명 |
|---|---|
| 위기 앵커 | 심리적 고통의 본질을 담은 4개 보편 문장(부정적 감정 : 4가지) → 위기 공간 구성 (추가적인 논의 필요)|
| Max-Pooling 유사도 | 일기 벡터 × 위기 앵커 → 일기별 최고 점수 추출 |
| 연속성 가중치 | 부정 감정 연속 시 `1.3 + stack × 0.2` 가속 |
| 응집도 보정 | centroid 기반 응집도 30% 반영 |
| 최종 공식 | `necessity_score = negativity_score × 0.7 + cohesion × 0.3` |


## Stage 2 — Context Extraction

트리거가 발동된 경우, Generation에 전달할 세 가지 컨텍스트를 추출합니다.

| 변수 | 추출 방법 |
|---|---|
| `dominant_emotion` | 7일 window 내 부정 감정 레이블 최빈값 |
| `key_context` | centroid와 코사인 유사도 최대인 일기 (이번 주 대표 일기) |
| `recent_diaries` | 최근 7일치 날짜 + 일기 본문 |

## Stage 3 — Generation (LangChain LCEL)

추출된 컨텍스트를 LLM에 전달해 공감 메시지와 처방 조언을 생성합니다.

```
[입력]
    dominant_emotion  → 지배 감정 레이블
    key_context       → 핵심 맥락 일기 (날짜 + 내용)
    recent_diaries    → 최근 7일치 일기

[출력 구조]
    1. 공감  — 일기 내용을 근거한 구체적 공감 메시지
    2. 조언  — 지배 감정에서 벗어나기 위한 처방 1~2가지
```

In [7]:
class EmotionalTriggerAnalyzer:
    
    NEGATIVE_EMOTIONS = ["분노", "슬픔", "두려움", "불쾌함"]

    def __init__(self, api_key, model_name="text-embedding-3-small"):
        self.embeddings_model = OpenAIEmbeddings(openai_api_key=api_key, model=model_name)

    def load_diary_window(self, file_path, days=7):
        """최근 n일간의 일기 데이터를 로드하고 최신순으로 반환합니다."""
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        recent_dates = sorted(data.keys(), reverse=True)[:days]
        window_docs = []
        for date in recent_dates:
            entry = data[date]
            text = f"날짜: {entry['date']}, 감정: {entry['emotion']}, 일기: {entry['diary']}"
            window_docs.append({
                "date": entry["date"],
                "text": text,
                "emotion": entry["emotion"],
                "diary": entry["diary"],
            })
        return window_docs

    def _get_crisis_space(self):
        """심리적 고통의 본질을 담은 보편적 앵커 벡터를 반환합니다."""
        anchors = [
            "삶의 의욕이 전혀 없고 모든 것이 허무하게 느껴져 전문가의 상담이 필요하다.",
            "극심한 불안감과 중압감 때문에 일상생활을 유지하기가 힘들어 도움이 필요하다.",
            "혼자라는 고립감과 외로움이 깊어져 누구에게라도 마음을 털어놓고 싶다.",
            "정신적으로 한계에 다다른 것 같아 누군가의 개입이 절실한 상태이다.",
        ]
        return np.array(self.embeddings_model.embed_documents(anchors))

    def analyze_necessity(self, window_docs):
        """necessity_score, centroid, vectors를 반환합니다."""
        texts = [doc["text"] for doc in window_docs]
        vectors = np.array(self.embeddings_model.embed_documents(texts))

        crisis_space = self._get_crisis_space()
        similarity_matrix = cosine_similarity(vectors, crisis_space)  
        individual_max_scores = np.max(similarity_matrix, axis=1)     

        # 연속성 가중치 적용 - 지속적인 부정적 감정에 대한 추가적인 수치 부여
        weighted_scores = []
        stack = 0
        for score, doc in zip(individual_max_scores, window_docs):
            if doc["emotion"] in self.NEGATIVE_EMOTIONS:
                stack += 1
                weight = 1.3 + stack * 0.2
            else:
                stack = 0
                weight = 1.0
            weighted_scores.append(score * weight)

        negativity_score = np.mean(weighted_scores)
        centroid = np.mean(vectors, axis=0)
        cohesion = float(np.mean(cosine_similarity(vectors, [centroid])))
        necessity_score = negativity_score * 0.7 + cohesion * 0.3

        return necessity_score, centroid, vectors

    def extract_key_context(self, window_docs, vectors, centroid):
        """centroid와 가장 가까운 일기 = 이번 주를 대표하는 핵심 맥락을 반환합니다."""
        similarities = cosine_similarity(vectors, [centroid]).flatten()
        idx = np.argmax(similarities)
        return window_docs[idx]

print("EmotionalTrajectoryAnalyzer 클래스 정의 완료")

EmotionalTrajectoryAnalyzer 클래스 정의 완료


In [3]:
TARGET_USER = "김부장"
NEGATIVE_EMOTIONS = ["분노", "슬픔", "두려움", "불쾌함"]

In [4]:
import yaml
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

PROMPT_PATH = "prompts/proactive_doctor.yaml"

# YAML 로드
with open(PROMPT_PATH, "r", encoding="utf-8") as f:
    prompt_config = yaml.safe_load(f)

# Few-shot 프롬프트 구성
# 부정적인 감정의 항목 : NEGATIVE_EMOTIONS = ["분노", "슬픔", "두려움", "불쾌함"] 에 대해서 각각 하나씩, 총 4개 예시 작성.
example_prompt = ChatPromptTemplate.from_messages([
    ("human", prompt_config["human_template"]),
    ("ai", "{answer}"),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=prompt_config["few_shot_examples"],
)

# 최종 프롬프트 구성
final_prompt = ChatPromptTemplate.from_messages([
    ("system", prompt_config["system_prompt"]),
    few_shot_prompt,
    ("human", prompt_config["human_template"]),
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7, openai_api_key=OPENAI_API_KEY)

# LCEL Chain
doctor_chain = final_prompt | llm | StrOutputParser()

print(f"프롬프트 로드 완료: {PROMPT_PATH}")
print(f"  - system_prompt: {len(prompt_config['system_prompt'])}자")
print(f"  - few_shot_examples: {len(prompt_config['few_shot_examples'])}개")
print("감정 주치의 Few-shot Chain 정의 완료")

프롬프트 로드 완료: prompts/proactive_doctor.yaml
  - system_prompt: 1485자
  - few_shot_examples: 4개
감정 주치의 Few-shot Chain 정의 완료


## 전체 파이프라인 (Full Pipeline)

In [5]:
TARGET_USER = "김부장"
NEGATIVE_EMOTIONS = ["분노", "슬픔", "두려움", "불쾌함"]

In [8]:
def run_proactive_doctor(diary_path: str, threshold: float = 0.6) -> dict:
    """
    감정 주치의 전체 파이프라인을 실행합니다.

    Returns:
        triggered (bool), necessity_score (float), message (str | None)
    """
    # Trigger
    analyzer = EmotionalTriggerAnalyzer(api_key=OPENAI_API_KEY)
    window_docs = analyzer.load_diary_window(diary_path, days=7)
    score, centroid, vectors = analyzer.analyze_necessity(window_docs)

    if score <= threshold:
        return {"triggered": False, "necessity_score": score, "message": None}

    # Context Extraction
    negative_labels = [doc["emotion"] for doc in window_docs if doc["emotion"] in NEGATIVE_EMOTIONS]
    dominant_emotion = Counter(negative_labels).most_common(1)[0][0]

    key_doc = analyzer.extract_key_context(window_docs, vectors, centroid)
    key_context = f"[{key_doc['date']}] {key_doc['diary']}"

    recent_diaries = "\n".join(
        f"[{d['date']} / {d['emotion']}] {d['diary']}"
        for d in window_docs[:7]
    )

    # Stage 3: Generation
    message = doctor_chain.invoke({
        "dominant_emotion": dominant_emotion,
        "key_context": key_context,
        "recent_diaries": recent_diaries,
    })

    return {"triggered": True, "necessity_score": score, "message": message}


# 실행
result = run_proactive_doctor(f"data/{TARGET_USER}/diary.json")

print(f"트리거: {'ON ✅' if result['triggered'] else 'OFF'}")
print(f"necessity_score: {result['necessity_score']:.4f}")
if result["message"]:
    print("\n" + "=" * 60)
    print(result["message"])

트리거: ON ✅
necessity_score: 0.6295

**공감**
최근 일주일 동안 주말에도 업무 메일 소리에 불안한 마음이 가시지 않고, 사무실에서 느끼는 존재의 무게가 정말 무겁고 슬프셨겠어요. 박 부장의 퇴사로 인한 두려움과 함께 누구에게도 털어놓을 수 없는 무거운 마음이 뼈저리게 느껴지는 상황이군요. 그래도 3월 19일에 퇴근길에 개나리를 보며 봄을 느끼고, 아내가 해준 된장찌개로 잠시나마 편안함을 찾으셨던 순간이 있었어요. 그래도 여전히 가족들과의 거리감과 자신의 존재에 대한 의문이 남아 있을 것 같아요. 당분간은 그 편안했던 저녁처럼 작은 기쁨의 순간들을 떠올리며, 아래 조언들을 함께 해보아요.

**조언**
1. **감정 인식과 수용 연습 (ACT)**: 슬픔과 두려움의 감정을 느끼고 있는 현재의 자신을 인정해 주세요. 감정일기나 메모에 '나는 지금 슬프다', '나는 지금 두렵다'라고 적고, 이 감정을 잠시 느끼며 인정하는 시간을 가져보세요. 감정을 피하지 않고 수용하는 것이 자아를 회복하는 첫걸음입니다.
2. **지지 네트워크 구축**: 신뢰할 수 있는 친구나 가족과 함께 소통해 보세요. '요즘 힘들어'라는 메시지를 보내는 것으로 시작해 보세요. 누군가에게 털어놓는 것만으로도 마음이 가벼워질 수 있습니다. 전문가의 도움을 받는 것도 고려해보세요.
3. **소소한 즐거움 찾기**: 매일 작은 즐거움을 계획해보세요. 예를 들어, 퇴근 후 10분간 산책하거나 좋아하는 음악을 듣는 것처럼 일상에서 느낄 수 있는 소소한 기쁨을 찾아보세요. 이런 작은 습관들이 마음의 안정을 가져다 줄 수 있습니다.
